<a href="https://colab.research.google.com/github/llelus/DSA-Project/blob/main/notebooks/01_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Data Collection

Builds the raw dataset for the study window set in `src/config.py` (`START_DATE`–`END_DATE`, by market expiry):

| Output | Content |
|---|---|
| `data/raw/polymarket_markets.csv` | one row per hourly *"Bitcoin above K on …, H ET?"* market: strike, start/end, YES token, outcome |
| `data/raw/polymarket_prices/YYYY-MM-DD.parquet` | YES-price history at 1-minute fidelity, one file per expiry day |
| `data/raw/btc_1m/YYYY-MM-DD.parquet` | Binance BTC/USDT 1-minute klines (public archive, full history) |
| `data/processed/panel_1m.parquet` | the analysis panel (derived, git-ignored, rebuilt from raw) |

Every step is **resumable**: files that exist are skipped, so after a Colab disconnect just re-run the cells.
For long windows, run the push cell at the end now and then so the downloaded files are not lost.

In [ ]:
# Setup — identical in every notebook; works in Colab and locally
import os, sys, subprocess
if "google.colab" in sys.modules:
    REPO = "/content/DSA-Project"
    if os.path.exists(REPO):
        subprocess.run(["git", "-C", REPO, "pull", "-q", "--autostash"])
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/llelus/DSA-Project.git", REPO], check=True)
    os.chdir(REPO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import pandas as pd
from src import config, collect, dataset
from src.results import save_metrics

session = collect.make_session()
print("Study window:", config.START_DATE, "->", config.END_DATE)

## 1. Probe one day

Before the full download, check that the API returns the markets we expect, that `MARKET_PATTERN`
matches their wording, and read the **resolution rules** (which exchange/candle settles the market).
If nothing matches, adjust `MARKET_PATTERN` or the window in `src/config.py`.

In [ ]:
day0 = pd.Timestamp(config.START_DATE, tz="UTC")
raw = collect.fetch_markets_page(session, day0, day0 + pd.Timedelta(days=1), offset=0)
btc_q = [m.get("question", "") for m in raw if "bitcoin" in m.get("question", "").lower()]
matched = [m for m in raw if collect.parse_market(m)]
print(f"{day0.date()}: {len(raw)} markets returned, {len(btc_q)} mention Bitcoin, {len(matched)} match MARKET_PATTERN")
print("Sample Bitcoin questions:", *btc_q[:10], sep="\n  ")
if matched:
    print("\nResolution rules of a matched market:\n", matched[0].get("description", "")[:1200])

## 2. Market list

In [ ]:
markets = collect.fetch_markets(config.START_DATE, config.END_DATE, session=session)
if markets.empty:
    raise SystemExit("No markets found: adjust START_DATE/END_DATE or MARKET_PATTERN in src/config.py")
markets = collect.save_markets(markets)
life_min = (markets.market_end - markets.market_start).dt.total_seconds() / 60
print(f"{len(markets):,} markets saved -> {config.RAW_MARKETS}")
print(f"Lifetime (minutes): median {life_min.median():.0f}, 90th pct {life_min.quantile(.9):.0f}")
print("Outcome balance:", markets.result_yes_won.value_counts(dropna=False).to_dict())
markets.head()

## 3. BTC spot — Binance BTC/USDT 1-minute klines

In [ ]:
missing_btc_days = collect.collect_btc(config.START_DATE, config.END_DATE, session=session)

## 4. Polymarket price histories (longest step)

In [ ]:
price_summary = collect.collect_prices(markets, session=session, workers=4)

## 5. Data-quality checks

1. **Settlement rule** — recompute every outcome as *settlement candle close > strike*. Accuracy close to 100%
   confirms that our BTC series is the resolution source and which candle settles the market.
   If the best offset differs from `SETTLE_CANDLE_OFFSET_MIN` in `src/config.py`, change it there.
2. **Price resolution** — spacing between Polymarket price points (v1 data was 10-minute, which made
   minute-level lag claims impossible).
3. **Coverage** — markets without any price history.

In [ ]:
btc = collect.load_btc()
prices = collect.load_prices()

check = dataset.resolution_check(markets, btc, offsets=(-1, 0, 1))
print(check.to_string(index=False))
best = check.loc[check.accuracy.idxmax()]
print(f"Best settlement offset: {int(best.offset_min):+d} min ({best.accuracy:.2%}); "
      f"config uses {config.SETTLE_CANDLE_OFFSET_MIN:+d}")

spacing = prices.groupby("condition_id")["ts"].diff().dt.total_seconds() / 60
print("\nPolymarket point spacing (min):", spacing.describe(percentiles=[.1, .5, .9]).round(2).to_dict())
covered = markets.condition_id.isin(prices.condition_id.unique())
print(f"Markets with price history: {covered.sum():,}/{len(markets):,} ({covered.mean():.1%})")

## 6. Build the analysis panel

In [ ]:
panel = dataset.load_panel(rebuild=True)
print(f"Panel: {len(panel):,} market-minutes, {panel.condition_id.nunique():,} markets")
print(f"Share of minutes with a fresh Polymarket price (<1 min old): {(panel.stale_min < 1).mean():.1%}")
print(f"Rows with fair value defined: {panel.fair.notna().mean():.1%}")

save_metrics("data",
    window=[config.START_DATE, config.END_DATE],
    n_markets=len(markets), n_markets_with_prices=int(covered.sum()),
    n_price_points=len(prices), median_point_spacing_min=float(spacing.median()),
    settle_offset_best=int(best.offset_min), settle_accuracy_best=float(best.accuracy),
    settle_accuracy_config=float(check.set_index("offset_min").loc[config.SETTLE_CANDLE_OFFSET_MIN, "accuracy"]),
    n_btc_days_missing=len(missing_btc_days),
    panel_rows=len(panel), panel_markets=int(panel.condition_id.nunique()),
    share_fresh_price=float((panel.stale_min < 1).mean()))

In [ ]:
# Optional: commit outputs back to GitHub (needs Colab secrets GITHUB_TOKEN, GIT_NAME, GIT_EMAIL)
PUSH = False
if PUSH:
    from src.colab import push
    push("data: collect raw Polymarket + BTC data", paths=('data/raw', 'results'))